# Manager Agent — intake / front door (RAG contact flow)

The user says who they are meeting and anything they already know; a contacts store
(RAG, here stubbed) returns the stored profile; the Manager produces `subjects`
(`name` + `phrase`), the `relationship`, and an advice-free `overall_context`.
The utterance is PRIMARY; the profile SUPPLEMENTS it. `%run common.ipynb` for the bootstrap.

In [ ]:
import os
# Run from backend/ so `%run common.ipynb` resolves regardless of the kernel's
# launch dir (IDE/Jupyter kernels often start at the workspace root).
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
%run common.ipynb


In [ ]:
# Manager config + schemas (manager-specific; not in common).
from typing import Literal

MANAGER_MODEL  = os.getenv("MANAGER_MODEL",  "openai/gpt-4.1")
MANAGER_EFFORT = os.getenv("MANAGER_EFFORT", "none")
MAX_SUBJECTS   = 8


class Subject(BaseModel):
    name: str = Field(
        description="Short canonical subject label, e.g. 'Italy', 'rock climbing', 'World Cup'."
    )
    phrase: str = Field(
        description="A short phrase describing how this subject connects to the person, "
        "e.g. 'recently been to Italy'. Copied from the user's own words when the utterance "
        "raises the subject; otherwise drawn faithfully from the retrieved contact profile. "
        "This is what lets the next stage write personal angles instead of generic trivia. "
        "Describes ONLY this one subject; never bundles in a different subject."
    )


Relationship = Literal[
    "friend", "coworker", "boss", "report",
    "first-date", "partner", "family", "acquaintance",
]


class ManagerOutput(BaseModel):
    subjects: list[Subject] = Field(
        description="Distinct topics worth discussing, utterance subjects first then profile "
        "supplements. Empty if neither utterance nor profile raises any."
    )
    relationship: Relationship = Field(
        default="friend",
        description="Best-guess relationship between the user and the person they are meeting. "
        "Default to 'friend' when neither utterance nor profile clearly indicates another relationship.",
    )
    overall_context: str = Field(
        description="1-2 plain sentences describing who the user is meeting and the situation. "
        "Description only: no advice, no instructions to downstream agents."
    )


print("Subject, Relationship, ManagerOutput defined.")


In [3]:
# Cell 3 — Manager agent: utterance (primary) + retrieved profile (supplement) -> ManagerOutput

MANAGER_PROMPT = f"""\
You are the intake step of a conversation-prep tool. The user is about to meet
someone. Your job is to produce the structured facts the rest of the pipeline
needs: the subjects worth talking about, the relationship, and a short
advice-free description of the meeting. You do no research and you give no advice.

For reference, today's date is {TODAY}.

You will receive an input message with two parts:
  Utterance:    <what the user typed, e.g. 'I am meeting Sarah, she just got into pottery'>
  Contact info: <profile text retrieved about this person, or the word 'none'>

PRIORITY — the utterance comes first:
- Utterance is the PRIMARY source. It gives the SITUATION (coffee, dinner, first
  date) AND any interests or topics the user explicitly mentions. Whatever the
  user says here takes priority: it is what they know right now, which may be
  fresher and more relevant than the stored profile.
- Contact info, when present, is the SUPPLEMENTARY source. It is a stored profile
  of the person. Use it to ADD subjects the utterance did not already cover.

Extract THREE things into ManagerOutput:

1. subjects — the distinct topics worth discussing, gathered in this order:
   a) FIRST, every interest or topic the UTTERANCE raises, in the user's own words.
   b) THEN, additional subjects from the Contact info profile that the utterance
      did not already cover (interests, hobbies, work, recent events).
   When the same subject appears in both, keep ONE entry and use the utterance's
   phrasing — the user's own words win.
   For each subject:
     name    A short canonical label, e.g. 'rock climbing', 'Japan', 'F1'.
     phrase  A short natural phrase describing how the subject connects to the
             person. When the subject came from the utterance, copy the user's
             phrasing. When it came only from the profile, write the phrase
             FAITHFULLY from the profile — do not exaggerate or invent detail.
             The phrase is what lets the next stage write personal angles
             instead of generic trivia.
             Each phrase must describe ONLY its own subject — never bundle a
             different subject into one phrase (split "likes soccer and just had
             a Boston internship" into two separate subjects).
   Keep subjects DISTINCT. Merge near-synonyms, a subtype with its parent, or
   topics drawn from the same clause into ONE subject — the next stage already
   explores facets within a subject. When you merge, keep ALL the specifics in
   the phrase so the detail is not lost. E.g. "rock climbing and bouldering" ->
   name "rock climbing", phrase "is really into rock climbing, especially
   bouldering"; "hiking with her two dogs" -> name "hiking", phrase "loves
   hiking with her two dogs".
   Only use topics actually present in the utterance or profile. Never invent.
   If there are more than {MAX_SUBJECTS} subjects, keep the {MAX_SUBJECTS} most
   distinct ones (utterance subjects first) and drop the rest.

2. relationship — who this person is to the user, chosen from this EXACT list:
     friend, coworker, boss, report, first-date, partner, family, acquaintance
   Infer it from the utterance and the profile. DEFAULT to 'friend' when neither
   clearly indicates another relationship. Do not overthink this.

3. overall_context — 1-2 plain sentences describing who the user is meeting and
   the situation, fusing the occasion (from the utterance) with who the person is
   (from the profile). Describe ONLY. No advice, no suggestions, and no
   instructions to other agents (never write things like 'you should ask about...').

If there is no profile and the utterance raises no real subject, return an empty
subjects list.

---

EXAMPLE

Input:
  Utterance:    I'm grabbing coffee with Sarah this weekend, she's been really into baking sourdough lately
  Contact info: Sarah is a close friend from university. She works as a data
  scientist, is really into bouldering, and just got back from a trip to Japan.
  She has also been learning to bake sourdough.

Output:
{{
  "subjects": [
    {{"name": "sourdough baking", "phrase": "has been really into baking sourdough lately"}},
    {{"name": "data science", "phrase": "works as a data scientist"}},
    {{"name": "bouldering", "phrase": "is really into bouldering"}},
    {{"name": "Japan", "phrase": "just got back from a trip to Japan"}}
  ],
  "relationship": "friend",
  "overall_context": "The user is grabbing coffee this weekend with Sarah, a close friend from university who works as a data scientist."
}}
"""

manager_agent = Agent(
    name="ManagerAgent",
    instructions=MANAGER_PROMPT,
    model=model(MANAGER_MODEL),
    model_settings=model_settings(MANAGER_EFFORT),
    output_type=ManagerOutput,
)


async def run_manager(user_input: str, contact_info: str | None = None) -> ManagerOutput:
    """Utterance (primary) + retrieved profile (supplement) -> ManagerOutput. No tools, single turn."""
    msg = f"Utterance: {user_input}\nContact info: {contact_info or 'none'}"
    result = await Runner.run(manager_agent, msg, max_turns=2)
    return result.final_output


def show(mo: ManagerOutput) -> None:
    print(f"relationship: {mo.relationship}")
    print(f"overall_context: {mo.overall_context}")
    print(f"subjects ({len(mo.subjects)}):")
    for s in mo.subjects:
        print(f"  - name='{s.name}'  phrase='{s.phrase}'")


print("manager_agent + run_manager + show defined.")


manager_agent + run_manager + show defined.


In [4]:
# Cell 4 — Contacts store (stand-in for the RAG vector store) + prep() entry point
#
# In production this is a vector DB keyed by contact; retrieval returns the
# stored profile text for the named person. Here a dict + naive name match
# stands in, so we can test the Manager's utterance-first + profile-supplement logic.

CONTACTS = {
    "sarah": "Sarah is a close friend from university. She works as a data scientist "
             "and is really into rock climbing and bouldering. She recently got back "
             "from a two-week trip to Japan, and has been learning to bake sourdough. "
             "She follows Formula 1 closely and never misses a race.",
    "david": "David is my manager at work, head of the engineering team. He is an avid "
             "marathon runner, a serious specialty-coffee enthusiast who roasts his own "
             "beans, and recently became a father to a baby girl.",
    "mia":   "Mia is someone I matched with on a dating app; this is our first date. "
             "Her profile says she is a veterinarian, loves hiking with her two dogs, "
             "plays the cello, and is a big fan of science fiction novels.",
}


def retrieve_contact(user_input: str) -> str | None:
    """Stand-in for RAG retrieval: return the profile of a known contact named
    in the utterance, or None if no contact matches."""
    low = user_input.lower()
    for name, profile in CONTACTS.items():
        if name in low:
            return profile
    return None


async def prep(user_input: str) -> ManagerOutput:
    """Entry point: 'I am meeting XXX' -> retrieve profile -> Manager rewrite."""
    profile = retrieve_contact(user_input)
    print(f"  [retrieval] {'hit' if profile else 'miss -> utterance-only fallback'}")
    return await run_manager(user_input, profile)


print("CONTACTS store + retrieve_contact + prep defined.")


CONTACTS store + retrieve_contact + prep defined.


In [ ]:
# Demo / tests (standalone only) — gated so `%run` from the pipeline stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    CASES = [
        "I'm meeting Sarah this weekend, she's been getting really into pottery lately",
        "dinner with David tonight, he just finished his first marathon!",
        "first date with Mia on Friday",
        "meeting a friend who is interested in stock market, and is gonna see the world cup",
    ]
    for utt in CASES:
        print("=" * 64); print(utt)
        show(await prep(utt))
        print()
